#### Jointure spikes/LFP

LFP_spikes/
└── common_preprocess/
    ├── P119_FM71_stim4/                           {créé quand on run seulement une session}
    │   ├── P119_FM71_stim4_common_trials.csv
    │   └── P119_FM71_stim4_common_metadata.json
    
    ├── run_all_common_preprocess_sessions.csv     {créé quand on run tout en batch}
    └── run_all_common_preprocess_summary.json     {créé quand on run tout en batch}

##### une session

In [ ]:
# jointure spike/lfp sur une session

from utils_spike_lfp.spike_lfp_common_preprocess_session import (
    CommonPreprocessConfig,
    prepare_common_session,
    save_common_session_bundle,
    load_common_session_bundle,
    validate_common_trials,
)

micro_root="/media/aube/Aube/Spike-sorting"  # contient Data_folders/...
macro_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP" # contient .TRC + events macro
hilbert_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert" 
common_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/common_preprocess" 

cfg = CommonPreprocessConfig(
    micro_root=micro_root,
    macro_root=macro_root,
    hilbert_root=hilbert_root,
    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,
    event_match_strategy="auto",
    compute_micro_macro_offset_qc=True,
)

bundle = prepare_common_session(
    patient="P119_FM71",
    session=4,
    cfg=cfg,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
)

qc = validate_common_trials(bundle.trials)
print(qc)

session_common_dir = save_common_session_bundle(bundle, common_root)

bundle2 = load_common_session_bundle(
    session_common_dir,
    load_hilbert=True,
    hilbert_bands=("theta", "alpha", "beta")
)



=== Base commune spike-LFP | P119_FM71_stim4 ===
[INFO] offset micro↔macro: offset=-41.471000s, MAD résidus=0s, max|résidu|=1.13687e-13s, n_used=31/31
[INFO] Hilbert exports chargés: ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
{'n_trials': 31, 'missing_required_columns': [], 'nan_counts': {'t_start_micro': 0, 't_end_micro': 0, 't_start_macro': 0, 't_end_macro': 0}, 'n_label_order_mismatch': 0, 'group_counts': {'unknown': 25, 'cog+': 3, 'negatif': 2, 'controle': 1}, 'n_unparsed_stim_labels': 0}


##### toutes les sessions

In [ ]:
# jointure spike/lfp sur toutes les sessions avec hilbert_stats fait et nwb dispo

from utils_spike_lfp.spike_lfp_common_preprocess_batch import (
    CommonBatchPreprocessConfig,
    run_all_common_preprocess,
)

cfg_common = CommonBatchPreprocessConfig(
    micro_root="/media/aube/Aube/",
    macro_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP",
    hilbert_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP/theta_gamma_cog/results_hilbert",
    common_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/common_preprocess",

    session_source="hilbert",
    require_hilbert_exports=True,
    require_existing_nwb=True,

    pre_length=3.0,
    post_length=3.0,
    epsilon=0.1,

    overwrite=False,
    verbose=True,
)

out_common = run_all_common_preprocess(cfg_common)


=== Common batch | P119_FM71_stim4 ===

=== Base commune spike-LFP | P119_FM71_stim4 ===
[INFO] offset micro↔macro: offset=-41.471000s, MAD résidus=0s, max|résidu|=1.13687e-13s, n_used=31/31
[INFO] Hilbert exports chargés: ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']


#### Spike / power

results_fr_power_corr/
├── fr_logratio__lfp_pre_logratio/
    ├── per_session/
        └── P119_FM71_stim4/                                         {créé}
            ├── figures_significant_histograms/
            ├── P119_FM71_stim4_fr_power_corr_config.json
            ├── P119_FM71_stim4_fr_bins.csv
            ├── P119_FM71_stim4_fr_power_trial_bins_theta.csv       => une ligne correspond à unit × stim × channel × bin temporel, dans cette bande de fréquence
            ├── P119_FM71_stim4_fr_power_trial_bins_alpha.csv
            ├── P119_FM71_stim4_fr_power_trial_bins_beta.csv
            ├── P119_FM71_stim4_fr_power_trial_bins_low_gamma.csv
            ├── P119_FM71_stim4_fr_power_trial_bins_high_gamma.csv
            ├── P119_FM71_stim4_fr_power_corr_run_summary.json
            └── P119_FM71_stim4_fr_power_correlations.csv            {table principale} => une ligne est une corrélation across trials *

    └── pooled_across_sessions/
        ├── figures_significant_histograms/
        ├── ALL_SESSIONS_fr_power_correlations_pooled.csv
        ├── ALL_SESSIONS_fr_power_correlations_pooled_SIGNIF.csv
        ├── ALL_SESSIONS_fr_power_pool_summary.json

        ├── run_all_fr_power_correlations_summary.json
        └── run_all_fr_power_correlations_sessions.csv

├── fr_zscore__lfp_pre_zscore/
    ├── per_session/ ...
    └── pooled_across_sessions/ ...

└── run_all_fr_power_variants_summary.csv


note : * c'est-à-dire que pour : 
unit 12
channel CU1-CU2
band theta
bin 0.5–0.6 s
corr_grouping = group_label_x_locality
corr_condition = cog+::local
=> Pour cette unit + ce canal +, cette bande + ce bin temporel, on prend uniquement les trials cog+ où le canal macro est local, puis on corrèle FR_norm avec LFP_power_norm à travers ces trials.

##### toutes les sessions

In [ ]:
# pour calcul spike/power sur toutes les sessions avec hilbert_stats fait et nwb dispo, et common_preprocess fait

from utils_spike_lfp.spike_lfp_power_correlations import (
    FRPowerCorrelationConfig,
    PooledFRPowerCorrelationVariantsConfig,
    run_all_available_sessions_fr_power_correlation_variants,
)

base_cfg = FRPowerCorrelationConfig(
    bin_width_ms_options=(100.0, 500.0),
    windows_to_correlate=("post",),

    fr_norm_method="logratio",
    lfp_value_col="lfp_power_pre_logratio",

    correlation_methods=("spearman",),

    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
    localities_to_include=("local", "distant"),

    smoothing_method="boxcar_bin",

    use_deadfiles=True,
    exclude_bins_overlapping_dead=True,
    min_valid_bin_fraction=1.0,

    save_trial_bin_tables=True,
    save_fr_table=True,
    save_lfp_table=False,

    save_significant_correlations=True,
    significance_p_col="p_value",
    significance_alpha=0.05,

    make_significant_histograms=True,
    histogram_method="spearman",
    histogram_significance_col="p_value",
    histogram_alpha=0.05,

    overwrite=True,
    reuse_existing_if_available=False,

    verbose=True,
)

pool_cfg = PooledFRPowerCorrelationVariantsConfig(
    common_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/common_preprocess",
    root_micro="/media/aube/Aube/",
    output_root="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/results_fr_power_corr",

    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),

    base_session_cfg=base_cfg, # on reprend les paramètres de base, mais on peut remplacer fr_norm_methods et lfp_value_cols par rapport aux valeurs initiales dans base_cfg

    fr_norm_methods=("logratio", "zscore"),
    lfp_value_cols=("lfp_power_pre_logratio", "lfp_power_pre_zscore", "lfp_power_bin_mean"),
    correlation_methods=("spearman",),

    require_existing_nwb=True,

    overwrite_session_outputs=True, # mettre False si c'est seulement pour inclure des sessions supplémentaires sans toutes les refaire

    recompute_pooled_fdr=True,
    verbose=True,
)

out = run_all_available_sessions_fr_power_correlation_variants(pool_cfg) # 70 min pour 4 méthodes, sur une session


=== FR-power multi-session | P119_FM71_stim4 ===
/media/aube/Aube/Spike-sorting/Data_folders/P119_FM71/P119_FM71_stim4/ .nwb existe deja ? True
[INFO] P119_FM71_stim4: calcul FR biné pour 5 unités
[INFO] P119_FM71_stim4: binning LFP band=theta
[INFO] P119_FM71_stim4: fusion FR×LFP band=theta
[INFO] P119_FM71_stim4: corrélations band=theta
[INFO] P119_FM71_stim4: binning LFP band=alpha
[INFO] P119_FM71_stim4: fusion FR×LFP band=alpha
[INFO] P119_FM71_stim4: corrélations band=alpha
[INFO] P119_FM71_stim4: binning LFP band=beta
[INFO] P119_FM71_stim4: fusion FR×LFP band=beta
[INFO] P119_FM71_stim4: corrélations band=beta
[INFO] P119_FM71_stim4: binning LFP band=low_gamma
[INFO] P119_FM71_stim4: fusion FR×LFP band=low_gamma
[INFO] P119_FM71_stim4: corrélations band=low_gamma
[INFO] P119_FM71_stim4: binning LFP band=high_gamma
[INFO] P119_FM71_stim4: fusion FR×LFP band=high_gamma
[INFO] P119_FM71_stim4: corrélations band=high_gamma
[OK] P119_FM71_stim4: corrélations sauvegardées -> /home/a

##### une session

In [ ]:
# pour calcul spike/power sur une seule session : 

from utils_spike_lfp.spike_lfp_common_preprocess_session import load_common_session_bundle, get_nwb
from utils_spike_lfp.spike_lfp_power_correlations import (
    make_units_dict_from_spikes_object,
    prepare_unit_metadata_and_dead_intervals,
    run_session_fr_power_correlations,
)

patient = "P119_FM71"
session = "4"
root_micro = "/media/aube/Aube/"

bundle = load_common_session_bundle( # on charge le common_preprocess de cette session
    "/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/common_preprocess/P119_FM71_stim4",
    load_hilbert=True,
    hilbert_bands=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
)

base_cfg = FRPowerCorrelationConfig(
    bin_width_ms_options=(100.0, 500.0),
    windows_to_correlate=("post",),

    fr_norm_method="logratio",
    lfp_value_col="lfp_power_pre_logratio",

    correlation_methods=("spearman",),

    bands_to_test=("theta", "alpha", "beta", "low_gamma", "high_gamma"),
    localities_to_include=("local", "distant"),

    smoothing_method="boxcar_bin",

    use_deadfiles=True,
    exclude_bins_overlapping_dead=True,
    min_valid_bin_fraction=1.0,

    save_trial_bin_tables=True,
    save_fr_table=True,
    save_lfp_table=False,

    save_significant_correlations=True,
    significance_p_col="p_value",
    significance_alpha=0.05,

    make_significant_histograms=True,
    histogram_method="spearman",
    histogram_significance_col="p_value",
    histogram_alpha=0.05,

    overwrite=True,
    reuse_existing_if_available=False,

    verbose=True,
)

spikes = get_nwb(patient, session, root_micro)
units = make_units_dict_from_spikes_object(spikes)

unit_metadata, dead_intervals_by_unit = prepare_unit_metadata_and_dead_intervals(
    patient=patient,
    session=session,
    root_micro=root_micro,
    spikes=spikes,
    use_deadfiles=True,
)

out = run_session_fr_power_correlations(
    bundle=bundle,
    units=units,
    out_dir="/home/aube/Documents/article_neuronal_stimic/effets_cog/LFP_spikes/results_fr_power_corr/fr_logratio__lfp_pre_logratio/per_session",
    cfg=base_cfg,
    dead_intervals_by_unit=dead_intervals_by_unit,
    unit_metadata=unit_metadata,
)